# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [ ]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

Now perform the same as before:
- Feature Scaling
- Feature Selection


In [ ]:
#your code here
spaceship.isnull().sum()

In [ ]:
spaceship.dropna(inplace=True)
spaceship.isnull().sum()

In [ ]:
#your code here
spaceship["Cabin"].value_counts()
spaceship['Cabin'] = spaceship['Cabin'].str.split('/').str[0]
spaceship['Cabin'].unique()

In [ ]:
spaceship = spaceship.drop(['PassengerId', 'Name'], axis=1)


In [ ]:
#your code here
from sklearn.datasets import fetch_openml
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.preprocessing import OneHotEncoder


In [ ]:
 #1. Initialize the encoder
encoder = OneHotEncoder(drop='first', sparse_output=False)

# 2. Fit and transform the column
encoded = encoder.fit_transform(spaceship[['HomePlanet']])

# 3. Create a DataFrame from the encoded array
encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(['HomePlanet'])
)

# 4. Concatenate with original DataFrame (or drop original if needed)
spaceship = pd.concat([spaceship.drop(columns=['HomePlanet']), encoded_df], axis=1)

#print(spaceship)

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


# List of categorical columns to encode
categorical_cols = ['CryoSleep', 'Cabin', 'Destination', 'VIP']

# Drop rows with missing values in categorical columns
spaceship_cleaned = spaceship.dropna(subset=categorical_cols)

# OneHotEncoder with drop='first' and handling unknown categories
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

# ColumnTransformer to apply encoding only to categorical columns
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', encoder, categorical_cols)
    ],
    remainder='passthrough'  # Leave other columns (like Age) untouched
)

# Transform the data
processed = preprocessor.fit_transform(spaceship_cleaned)

# Get column names after encoding
encoded_columns = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
all_columns = list(encoded_columns) + [col for col in spaceship_cleaned.columns if col not in categorical_cols]

# Create final DataFrame
spaceship_encoded = pd.DataFrame(processed, columns=all_columns)

print(spaceship_encoded)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Make a copy to avoid modifying the original
df = spaceship_encoded.copy()

# 2. Convert numeric columns to proper dtype
numeric_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# 3. Drop rows with missing target
df = df.dropna(subset=['Transported'])

# 4. Define X and y
X = df.drop('Transported', axis=1)
  # Convert to binary (1/0)
y = df['Transported'].astype(int)

# 5. Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train distribution:\n", y_train.value_counts(normalize=True))


In [ ]:
#your code here
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Initialize the Random Forest model
rf_clf = RandomForestClassifier(
    n_estimators=100,      # Number of trees in the forest
    max_depth=None,        # You can limit this to prevent overfitting
    random_state=42,
    n_jobs=-1              # Use all CPU cores
)

# Train the model
rf_clf.fit(X_train, y_train)

# Predict on the test set
y_pred_rf = rf_clf.predict(X_test)

# Evaluate
print("🌲 Random Forest Results:")
print(classification_report(y_test, y_pred_rf))
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.3f}")

- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

- Evaluate your model

**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [ ]:
#your code herefrom sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score

# Initialize the base Random Forest model
rf_clf = RandomForestClassifier(random_state=42, n_jobs=-1)

# Define the hyperparameter grid to search
param_grid = {
    'n_estimators': [100, 200, 300],  # Number of trees
    'max_depth': [None, 10, 20, 30],  # Maximum depth of trees
    'min_samples_split': [2, 5, 10],  # Minimum samples to split a node
    'min_samples_leaf': [1, 2, 4],    # Minimum samples at a leaf node
    'max_features': ['sqrt', 'log2'], # Number of features to consider at each split
    'bootstrap': [True, False]        # Whether to bootstrap samples
}



- Run Grid Search

In [ ]:
# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=rf_clf,
    param_grid=param_grid,
    cv=5,               # 5-fold cross-validation
    scoring='accuracy',  # Metric to optimize
    n_jobs=-1,          # Use all CPU cores
    verbose=2           # Print progress
)

# Fit GridSearchCV on training data
grid_search.fit(X_train, y_train)

# Get the best parameters and model
best_params = grid_search.best_params_
best_rf_clf = grid_search.best_estimator_

print("\n🔥 Best Hyperparameters:")
print(best_params)



- Evaluate your model

In [ ]:
# Evaluate the optimized model
y_pred_best = best_rf_clf.predict(X_test)

print("\n🎯 Optimized Random Forest Results:")
print(classification_report(y_test, y_pred_best))
print(f"Accuracy: {accuracy_score(y_test, y_pred_best):.3f}")